In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from matplotlib import rcParams
rcParams['axes.autolimit_mode'] = 'round_numbers'
rcParams['axes.xmargin'] = 0
rcParams['axes.ymargin'] = 0

In [ ]:
# Parameters
A = 1
α = 1/3
δ = 0.1
ρ = 0.04
β = 1/(1+ρ)
σ = 2
φ = 4

params = A, α, δ, ρ, β, σ, φ

In [ ]:
# Funkcje niezależne od podatków i funkcji użyteczności

def k_h(params):
    A, α, δ, ρ, β, σ, φ = params
    return (α*A/(δ+ρ))**(1/(1-α))

def y_h(params):
    A, α, δ, ρ, β, σ, φ = params
    return k_h(params)**α

def w(params):
    A, α, δ, ρ, β, σ, φ = params
    return (1-α)*y_h(params)

def c_h(params):
    A, α, δ, ρ, β, σ, φ = params
    return y_h(params) - δ*k_h(params)

In [ ]:
# Funkcje zależne od podatków i funkcji użyteczności

ψ = c_h(params)**(-σ) * w(params)		# Normalizacja: h=1 bez podatków

def h(params, ψ, τ_w=0, τ_c=0):
    A, α, δ, ρ, β, σ, φ = params
    try:
        return ( c_h(params)**(-σ) * w(params) * (1-τ_w)/(1+τ_c) / ψ )**(1/(φ+σ))
    except:
        return 0
    
h = np.vectorize(h, excluded=[0, 1])

In [ ]:
h(params, ψ)		# Sprawdzenie, czy h=1 przy braku podatków

In [ ]:
ττ = np.linspace(0, 1, 1+1000)		# Możliwe stawki podatkowe: od 0% do 100%

In [ ]:
# plt.subplots(figsize = (5, 3))

plt.plot(ττ*100, ττ*w(params)*h(params, ψ, ττ, 0), lw=3, label=r'$\tau^{w}$')		#'Podatek od dochodów z pracy')
plt.plot(ττ*100, ττ*c_h(params)*h(params, ψ, 0, ττ), lw=3, label=r'$\tau^{c}$')		#'Podatek konsumpcyjny')

plt.title('Przychody podatkowe')
plt.xlabel('Stawka opodatkowania (%)')
plt.legend(frameon=False, loc='upper left')
plt.show()

In [ ]:
# plt.subplots(figsize = (5, 3))

plt.plot(ττ*100, h(params, ψ, ττ, 0), lw=3, label=r'$\tau^{w}$')		#'Podatek od dochodów z pracy')
plt.plot(ττ*100, h(params, ψ, 0, ττ), lw=3, label=r'$\tau^{c}$')		#'Podatek konsumpcyjny')

plt.title('Przepracowane godziny na pracownika')
plt.xlabel('Stawka opodatkowania (%)')
plt.legend(frameon=False, loc='lower left')

plt.show()

In [ ]:
# plt.subplots(figsize = (5, 3))

plt.plot(ττ*w(params)*h(params, ψ, ττ, 0), 
         y_h(params)*h(params, ψ, ττ, 0), lw=3, label=r'$\tau^{w}$')		#'Podatek od dochodów z pracy')

plt.plot(ττ*c_h(params)*h(params, ψ, 0, ττ), 
         y_h(params)*h(params, ψ, 0, ττ), lw=3, label=r'$\tau^{c}$')		#'Podatek konsumpcyjny')

plt.title('PKB na pracownika')
plt.xlabel('Przychody podatkowe')
plt.legend(frameon=False, loc='lower right')

plt.yticks(np.arange(0.8, 1.8, 0.2))
plt.ylim(0.8, 1.6)


plt.show()

In [ ]:
# Replikacja Prescott (2004) Why do Americans Work so Much More than Europeans?

USA_h			= 1790 / 8766
USA_GDP_ph		= 59
USA_GDP_pc		= 102500
USA_τ_w			= 0.255

FR_DE_h			= 1465 / 8766
FR_DE_GDP_ph	= 56
FR_DE_GDP_pc	= 82200
FR_DE_τ_w		= 0.4375

# Kalibracja Prescotta (log-log)
σ = 1
φ = 0.5

# # Kalibracja "realistyczna" - jak na wykresach powyżej
# σ = 2
# φ = 4

In [ ]:
params_USA		= 1, α, δ, ρ, β, σ, φ
params_Europe	= FR_DE_GDP_ph/USA_GDP_ph, α, δ, ρ, β, σ, φ

USA_h_model = h(params_USA, ψ, USA_τ_w, 0)
FR_DE_h_model = h(params_Europe, ψ, FR_DE_τ_w, 0)

print('Przepracowane godziny: USA / Europa')
print('Model:', USA_h_model/FR_DE_h_model)
print('Dane: ', USA_h/FR_DE_h)

In [ ]:
USA_y_model = h(params_USA, ψ, USA_τ_w, 0)*y_h(params_USA)
FR_DE_y_model = h(params_Europe, ψ, FR_DE_τ_w, 0)*y_h(params_Europe)

print('PKB na pracownika: USA / Europa')
print('Model:', USA_y_model/FR_DE_y_model)
print('Dane: ', USA_GDP_pc/FR_DE_GDP_pc)

In [ ]:
print('Europa: wyniki w naturalnych jednostkach')
print('Roczne godziny pracy h:     ', 1790/(USA_h_model/FR_DE_h_model))
print('PKB na pracownika (PPP $) y:', USA_GDP_pc/(USA_y_model/FR_DE_y_model))